In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def start_driver(self):
    options = webdriver.ChromeOptions()
    options.add_argument("--user-data-dir=chrome-data")

    self.driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    self.driver.get("https://web.whatsapp.com/")
    print("Scan QR if needed...")
    time.sleep(20)

In [5]:
import time
import hashlib
import pandas as pd
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# ================= CONFIG =================
EXCEL_FILE_PATH = "Feb_surveillance.xlsx"
WHATSAPP_WEB = "https://web.whatsapp.com/"
WAIT_TIME = 20

# ================= CLASS =================
class WhatsAppMonitor:

    def __init__(self):
        self.driver = None
        self.processed_ids = set()

    # ---------- START DRIVER ----------
    def start_driver(self):
        options = Options()
        options.add_argument("--user-data-dir=chrome-data")  # keep login
        self.driver = webdriver.Chrome(options=options)
        self.driver.get(WHATSAPP_WEB)
        print("Scan QR if needed...")
        time.sleep(WAIT_TIME)

    # ---------- SAFE CLOSE ----------
    def close(self):
        if self.driver:
            self.driver.quit()

    # ---------- GENERATE UNIQUE ID ----------
    def generate_id(self, text):
        return hashlib.md5(text.encode()).hexdigest()

    # ---------- PARSE MESSAGE ----------
    def parse_message(self, text):
        try:
            lines = [l.strip() for l in text.split("\n") if l.strip()]

            data = {
                "date": None,
                "name": "",
                "message": text,
                "category": "General"
            }

            for line in lines:
                # detect date
                try:
                    parsed_date = datetime.strptime(line, "%d-%m-%Y")
                    data["date"] = parsed_date
                except:
                    pass

                # simple category detection
                if "biryani" in line.lower():
                    data["category"] = "Food"
                elif "complain" in line.lower():
                    data["category"] = "Complaint"

            return data

        except Exception as e:
            print("Parse error:", e)
            return None

    # ---------- GET MESSAGES ----------
    def get_messages(self):
        try:
            msgs = self.driver.find_elements(By.XPATH, "//div[contains(@class,'message')]")
            return msgs
        except:
            return []

    # ---------- SAVE TO EXCEL ----------
    def save_to_excel(self, records):
        try:
            df_new = pd.DataFrame(records)

            try:
                df_old = pd.read_excel(EXCEL_FILE_PATH)
                df = pd.concat([df_old, df_new], ignore_index=True)
            except:
                df = df_new

            df.to_excel(EXCEL_FILE_PATH, index=False)
            print("Saved to Excel")

        except Exception as e:
            print("Excel error:", e)

    # ---------- MAIN LOOP ----------
    def run(self):
        self.start_driver()

        try:
            while True:
                messages = self.get_messages()
                new_records = []

                for msg in messages:
                    text = msg.text.strip()
                    if not text:
                        continue

                    msg_id = self.generate_id(text)

                    if msg_id in self.processed_ids:
                        continue

                    self.processed_ids.add(msg_id)

                    parsed = self.parse_message(text)
                    if parsed:
                        new_records.append(parsed)

                if new_records:
                    self.save_to_excel(new_records)

                time.sleep(5)

        except KeyboardInterrupt:
            print("Stopped by user")

        finally:
            self.close()


# ================= RUN =================
if __name__ == "__main__":
    bot = WhatsAppMonitor()
    bot.run()

SessionNotCreatedException: Message: session not created: Chrome failed to start: crashed.
  (session not created: DevToolsActivePort file doesn't exist)
  (The process started from chrome location C:\Program Files\Google\Chrome\Application\chrome.exe is no longer running, so ChromeDriver is assuming that Chrome has crashed.); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6cbd429c5+2ed785]
	chromedriver!GetHandleVerifier [0x7ff6cba6a0d0+14e90]
	chromedriver!(No symbol) [0x7ff6cb7cdb2d]
	chromedriver!(No symbol) [0x7ff6cb80e742]
	chromedriver!(No symbol) [0x7ff6cb8097a1]
	chromedriver!(No symbol) [0x7ff6cb85dc51]
	chromedriver!(No symbol) [0x7ff6cb85d4b6]
	chromedriver!(No symbol) [0x7ff6cb819298]
	chromedriver!(No symbol) [0x7ff6cb81a183]
	chromedriver!GetHandleVerifier [0x7ff6cbd6de0d+318bcd]
	chromedriver!GetHandleVerifier [0x7ff6cbd685

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def start_driver(self):
    options = webdriver.ChromeOptions()
    options.add_argument("--user-data-dir=chrome-data")

    self.driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    self.driver.get("https://web.whatsapp.com/")
    print("Scan QR if needed...")
    time.sleep(20)

In [ ]:
import time
import hashlib
import pandas as pd
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from webdriver_manager.chrome import ChromeDriverManager


class WhatsAppBot:

    def __init__(self):
        self.driver = None
        self.processed = set()

    def start(self):
        options = Options()

        # 🔥 MOST IMPORTANT FIXES
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--remote-debugging-port=9222")

        # ❌ DO NOT use old chrome-data (causes crash)
        # options.add_argument("--user-data-dir=chrome-data")

        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=options
        )

        self.driver.maximize_window()
        self.driver.get("https://web.whatsapp.com/")

        print("👉 Scan QR code...")
        time.sleep(25)

    def generate_id(self, text):
        return hashlib.md5(text.encode()).hexdigest()

    def get_messages(self):
        try:
            msgs = self.driver.find_elements(By.XPATH, "//div[contains(@class,'message')]")
            return msgs
        except:
            return []

    def run(self):
        self.start()

        try:
            while True:
                messages = self.get_messages()

                for msg in messages:
                    text = msg.text.strip()

                    if not text:
                        continue

                    msg_id = self.generate_id(text)

                    if msg_id in self.processed:
                        continue

                    self.processed.add(msg_id)

                    print("📩 New Message:")
                    print(text)
                    print("-" * 40)

                time.sleep(5)

        except KeyboardInterrupt:
            print("Stopped")

        finally:
            if self.driver:
                self.driver.quit()


# RUN
if __name__ == "__main__":
    bot = WhatsAppBot()
    bot.run()
def parse_message(self, text):

    lines = text.strip().split("\n")

    # Extract parts safely
    time_val = lines[0].strip() if len(lines) > 0 else ""
    date_val = lines[1].strip() if len(lines) > 1 else ""
    branch = lines[2].strip() if len(lines) > 2 else ""

    # Remaining lines = complaint + response
    complaint = ""
    response = ""

    if len(lines) > 3:
        complaint = lines[3].strip()

    if len(lines) > 4:
        response = lines[4].strip()

    return {
        "Date": date_val,
        "Time": time_val,
        "Branch": branch,
        "Category": "",            # optional if not in text
        "Classification": "",
        "Stations": "",
        "Complaint": complaint,
        "Response": response,
        "Manager's Name": "",
        "Shift Manager": "",
        "Employee Name": ""
    }

👉 Scan QR code...
📩 New Message:
Hns Surveillance
Hns Surveillance
01:58pm
04-04-2026
Khadda Old dine in
No one stands here
0:06
2:07 PM
----------------------------------------
📩 New Message:
+92 328 2186043
Hns Surveillance
01:58pm
04-04-2026
Khadda Old dine in
No one stands here
0:10
2:08 PM
----------------------------------------
📩 New Message:
Hns Surveillance
0:33
02:02pm
04-04-2026
Khadda Old dine in
The customer is complaining that the  Platter two chicken is cold.
Edited2:15 PM
----------------------------------------
📩 New Message:
+92 343 2005449
Hns Surveillance
02:02pm
04-04-2026
Khadda Old dine in
The customer is complaining that the  Platter two chicken is cold.
Chef se Pata Kiya hai k yeh q tandi Gaye hai chicken tou us ka kehna tha k mein ne fresh uthar ker di hai baki mein ne us ko samjhaya hai k bilkul ghram Mal customer k pass jaye aur customer ko change ker k dosri chicken di hai..
2:26 PM
----------------------------------------
📩 New Message:
Muhammad Arbaz
+92 

In [7]:
def parse_message(self, text):

    lines = text.strip().split("\n")

    # Extract parts safely
    time_val = lines[0].strip() if len(lines) > 0 else ""
    date_val = lines[1].strip() if len(lines) > 1 else ""
    branch = lines[2].strip() if len(lines) > 2 else ""

    # Remaining lines = complaint + response
    complaint = ""
    response = ""

    if len(lines) > 3:
        complaint = lines[3].strip()

    if len(lines) > 4:
        response = lines[4].strip()

    return {
        "Date": date_val,
        "Time": time_val,
        "Branch": branch,
        "Category": "",            # optional if not in text
        "Classification": "",
        "Stations": "",
        "Complaint": complaint,
        "Response": response,
        "Manager's Name": "",
        "Shift Manager": "",
        "Employee Name": ""
    }